# TrojanLens — Colab (clean)

Real LoRA fine-tune of **Qwen2.5-Coder-0.5B** on the combined RS232+AES data, then
verified evaluation, a frozen-probe ablation, and a zero-shot baseline.

**Run order:** set GPU (Runtime → Change runtime type → **T4 GPU**). Run **Cell 1**,
then **Runtime → Restart session**, then run Cell 2 → end. At Cell 3 upload
`combined.jsonl`.


In [ ]:
# Cell 1 — install deps, then RESTART SESSION before Cell 2
!pip -q install "transformers>=4.44" "peft>=0.11" accelerate captum scikit-learn pyyaml 2>/dev/null
!pip uninstall -y torchao
print("deps installed -> now Runtime > Restart session, then run Cell 2 onward")

In [ ]:
# Cell 2 — GPU check
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
print(torch.cuda.get_device_name(0) if DEVICE=="cuda" else "No GPU: Runtime>Change runtime type>GPU")

In [ ]:
# Cell 3 — upload combined.jsonl
import os, json
if not os.path.exists("combined.jsonl"):
    from google.colab import files
    up = files.upload()
    n = list(up.keys())[0]
    if n != "combined.jsonl": os.rename(n, "combined.jsonl")
recs = [json.loads(l) for l in open("combined.jsonl") if l.strip()]
print("loaded", len(recs), "records;", sum(int(r['label']) for r in recs), "positive")

In [ ]:
# Cell 4 — config (0.5B pilot; raise model/seq on a bigger GPU)
import random, numpy as np, torch
CFG = {
    "model_name": "Qwen/Qwen2.5-Coder-0.5B",
    "dtype": torch.bfloat16, "max_seq_len": 1024,
    "lora": dict(r=16, alpha=32, dropout=0.05,
                 target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]),
    "epochs": 1, "lr": 1e-4,
    "focal_gamma": 2.0, "focal_alpha": 0.75, "loc_pos_weight": 8.0,
    "top_k": 15, "tau_g": 0.3, "tau_c": 0.05,
    "test_frac_variants": 0.25, "seed": 1234,
}
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
print("config ready:", CFG["model_name"], "| epochs", CFG["epochs"], "| seq", CFG["max_seq_len"])

In [ ]:
# Cell 5 — model (LoRA + heads; gradient checkpointing baked in)
import torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel
from peft import LoraConfig, get_peft_model

def focal_loss(logits, targets, gamma, alpha):
    ce = F.cross_entropy(logits, targets, reduction="none"); pt = torch.exp(-ce)
    at = torch.where(targets==1, torch.full_like(ce, alpha), torch.full_like(ce, 1-alpha))
    return (at*(1-pt)**gamma*ce).mean()

def aggregate_to_lines(vals, token_line, reduce="mean"):
    b={}
    for v,ln in zip(vals, token_line): b.setdefault(int(ln),[]).append(float(v))
    return {ln:(sum(x)/len(x) if reduce=="mean" else sum(x)) for ln,x in b.items()}

def neutralize(input_ids, token_line, lines, mask_id=0):
    m=set(int(x) for x in lines)
    return [mask_id if int(token_line[i]) in m else input_ids[i] for i in range(len(input_ids))]

class TrojanLens(nn.Module):
    def __init__(self, cfg, lora=True):
        super().__init__()
        enc = AutoModel.from_pretrained(cfg["model_name"], torch_dtype=cfg["dtype"])
        if lora:
            enc = get_peft_model(enc, LoraConfig(
                r=cfg["lora"]["r"], lora_alpha=cfg["lora"]["alpha"], lora_dropout=cfg["lora"]["dropout"],
                target_modules=cfg["lora"]["target_modules"], bias="none", task_type="FEATURE_EXTRACTION"))
            enc.enable_input_require_grads()
            enc.gradient_checkpointing_enable()   # fits 0.5B backprop on a T4
        else:
            for p in enc.parameters(): p.requires_grad_(False)
        self.encoder = enc
        self.hidden = getattr(getattr(enc,"config",None),"hidden_size",None) or enc.base_model.config.hidden_size
        self.detect = nn.Linear(self.hidden, 2)
        self.locate = nn.Linear(self.hidden, 1)
    def embedding_layer(self):
        e=self.encoder
        return e.get_input_embeddings() if hasattr(e,"get_input_embeddings") else e.base_model.get_input_embeddings()
    def encode(self, ids, attn):
        return self.encoder(input_ids=ids, attention_mask=attn).last_hidden_state
    def forward(self, ids, attn):
        if ids.dim()==1: ids=ids.unsqueeze(0); attn=attn.unsqueeze(0)
        h=self.encode(ids, attn); m=attn.unsqueeze(-1).to(h.dtype)
        pooled=(h*m).sum(1)/m.sum(1).clamp(min=1.0)
        return {"det": self.detect(pooled.float()), "tok": self.locate(h.float()).squeeze(-1)}
    def prob(self, ids, attn):
        with torch.no_grad():
            return torch.softmax(self.forward(ids, attn)["det"], -1)[0,1].item()
print("model class ready")

In [ ]:
# Cell 6 — variant-level train/test split (unseen Trojans)
def variant_of(f): return f.replace("\\","/").split("/")[0]
pos_vars = sorted(set(variant_of(r["file"]) for r in recs if int(r["label"])==1))
random.shuffle(pos_vars)
n_test = max(2, int(len(pos_vars)*CFG["test_frac_variants"]))
test_vars = set(pos_vars[:n_test])
train = [r for r in recs if variant_of(r["file"]) not in test_vars]
test  = [r for r in recs if variant_of(r["file"]) in test_vars]
print(f"{len(pos_vars)} trojan variants | test: {sorted(test_vars)}")
print(f"train {len(train)} ({sum(int(r['label']) for r in train)} pos) | test {len(test)} ({sum(int(r['label']) for r in test)} pos)")

In [ ]:
# Cell 7 — train (LoRA fine-tune or frozen probe)
def train_model(cfg, train_recs, lora=True):
    net = TrojanLens(cfg, lora=lora).to(DEVICE); net.train()
    params = [p for p in net.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"])
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([cfg["loc_pos_weight"]], device=DEVICE))
    print(f"trainable params: {sum(p.numel() for p in params)/1e6:.2f}M ({'LoRA+heads' if lora else 'heads only'})")
    for ep in range(cfg["epochs"]):
        random.shuffle(train_recs); tot=0.0
        for r in train_recs:
            ids = torch.tensor(r["input_ids"][:cfg["max_seq_len"]], dtype=torch.long, device=DEVICE)
            attn = torch.ones_like(ids); out = net(ids, attn)
            y = torch.tensor([int(r["label"])], device=DEVICE)
            ld = focal_loss(out["det"], y, cfg["focal_gamma"], cfg["focal_alpha"])
            tl = r["token_line"][:cfg["max_seq_len"]]; tset=set(int(x) for x in r["trojan_lines"])
            tgt = torch.tensor([1.0 if int(x) in tset else 0.0 for x in tl], device=DEVICE)
            ll = bce(out["tok"][0][:len(tgt)], tgt)
            opt.zero_grad(); (ld+ll).backward(); opt.step(); tot += float(ld+ll)
        print(f"  epoch {ep+1}/{cfg['epochs']}  avg loss {tot/max(1,len(train_recs)):.4f}")
    net.eval(); return net

print("Fine-tuning TrojanLens (~5-10 min on T4)...")
model = train_model(CFG, train, lora=True)

In [ ]:
# Cell 8 — attribution (fast grad*input) + verification
def attribute(net, r):
    ids = torch.tensor(r["input_ids"][:CFG["max_seq_len"]], dtype=torch.long, device=DEVICE).unsqueeze(0)
    attn = torch.ones_like(ids); emb=net.embedding_layer(); cap={}
    def hook(_m,_i,o): o.requires_grad_(True); o.retain_grad(); cap["e"]=o; return o
    hd = emb.register_forward_hook(hook)
    try:
        net.zero_grad(set_to_none=True); out=net(ids, attn); out["det"][0,1].backward()
        sal = (cap["e"]*cap["e"].grad).sum(-1)[0].detach().float().cpu().tolist()
    finally:
        hd.remove()
    ls = aggregate_to_lines([abs(x) for x in sal], r["token_line"][:CFG["max_seq_len"]], "sum")
    return [int(l) for l,_ in sorted(ls.items(), key=lambda kv: kv[1], reverse=True)[:CFG["top_k"]]]

def verify(net, r, cited, topk):
    ids=r["input_ids"][:CFG["max_seq_len"]]; tl=r["token_line"][:CFG["max_seq_len"]]
    def p(seq):
        t=torch.tensor(seq, dtype=torch.long, device=DEVICE); return net.prob(t, torch.ones_like(t))
    base=p(ids); yc=set(cited); AG=len(set(topk)&yc)/max(1,len(yc))
    dc=base-p(neutralize(ids, tl, cited)); flip=(base>=0.5) and ((base-dc)<0.5)
    import random as _r
    others=[l for l in set(int(x) for x in tl) if l not in yc]
    ctrl=_r.sample(others, min(len(cited), len(others))) if others else []
    hold = p(neutralize(ids, tl, ctrl))>=0.5
    return dict(verified=bool(AG>=CFG["tau_g"] and dc>=CFG["tau_c"] and flip and hold),
                AG=AG, delta_c=dc, delta_s=base-p(neutralize(ids, tl, others)), flip=flip)
print("attribution + verification ready")

In [ ]:
# Cell 9 — evaluate + metrics (also aggregates Delta_c, Delta_s, AG)
def evaluate(net, test_recs):
    preds=[]
    for r in test_recs:
        ids=torch.tensor(r["input_ids"][:CFG["max_seq_len"]], dtype=torch.long, device=DEVICE)
        prob=net.prob(ids, torch.ones_like(ids)); yp=int(prob>=0.5)
        with torch.no_grad():
            tok=net(ids, torch.ones_like(ids))["tok"][0].float().cpu().tolist()
        lp=aggregate_to_lines([torch.sigmoid(torch.tensor(t)).item() for t in tok],
                              r["token_line"][:CFG["max_seq_len"]], "mean")
        pred_lines=[l for l,s in lp.items() if s>=0.5] or [int(l) for l,_ in sorted(lp.items(), key=lambda kv: kv[1], reverse=True)[:CFG["top_k"]]]
        ver=False; ag=dc=ds=None
        if yp==1 or int(r["label"])==1:
            topk=attribute(net, r); v=verify(net, r, pred_lines[:CFG["top_k"]], topk)
            ver=v["verified"]; ag=v["AG"]; dc=v["delta_c"]; ds=v["delta_s"]
        preds.append(dict(y_true=int(r["label"]), y_pred=yp,
                          gt=set(int(x) for x in r["trojan_lines"]), pred=set(int(x) for x in pred_lines),
                          verified=ver, AG=ag, dc=dc, ds=ds))
    return preds

def metrics(preds):
    tp=fp=fn=tn=0
    for p in preds:
        if p["y_true"] and p["y_pred"]: tp+=1
        elif p["y_pred"] and not p["y_true"]: fp+=1
        elif p["y_true"] and not p["y_pred"]: fn+=1
        else: tn+=1
    prec=tp/(tp+fp) if tp+fp else 0; rec=tp/(tp+fn) if tp+fn else 0
    f1=2*prec*rec/(prec+rec) if prec+rec else 0
    pos=[p for p in preds if p["y_true"]]
    plc=sum(1 for p in pos if p["gt"] and len(p["gt"]&p["pred"])/len(p["gt"])>=0.5)/max(1,len(pos))
    iou=sum((len(p["gt"]&p["pred"])/len(p["gt"]|p["pred"])) if (p["gt"]|p["pred"]) else 0 for p in pos)/max(1,len(pos))
    pp=[p for p in preds if p["y_pred"]]; vr=sum(1 for p in pp if p["verified"])/max(1,len(pp))
    ags=[p["AG"] for p in preds if p["AG"] is not None]
    dcs=[p["dc"] for p in preds if p["dc"] is not None]
    dss=[p["ds"] for p in preds if p["ds"] is not None]
    mean=lambda a: round(sum(a)/len(a),3) if a else 0.0
    return dict(precision=round(prec,3), recall=round(rec,3), f1=round(f1,3), PLC=round(plc,3),
                IoU=round(iou,3), VR=round(vr,3), AG=mean(ags), delta_c=mean(dcs), delta_s=mean(dss),
                tp=tp, fp=fp, fn=fn, tn=tn)

m_ft = metrics(evaluate(model, test))
print("FINE-TUNED:", m_ft)

In [ ]:
# Cell 10 — frozen-probe ablation (memory-safe: frees the fine-tuned model first)
import gc
try: del model
except NameError: pass
gc.collect(); torch.cuda.empty_cache()
print("Training frozen-probe baseline (heads only, no LoRA)...")
probe = train_model(CFG, train, lora=False)
m_probe = metrics(evaluate(probe, test))
print("FROZEN PROBE:", m_probe)

In [ ]:
# Cell 11 — zero-shot baseline (prompting, no training)
import gc
for _v in ["model","probe"]:
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
ZS="Qwen/Qwen2.5-Coder-0.5B-Instruct"
zt=AutoTokenizer.from_pretrained(ZS)
zl=AutoModelForCausalLM.from_pretrained(ZS, torch_dtype=torch.bfloat16).to(DEVICE).eval()
def zs_pred(r, mx=120):
    src="\n".join(r["lines"][:mx])
    pr=("You are a hardware-security auditor. Does the following Verilog module "
        "contain a hardware Trojan (a hidden trigger plus a malicious payload)? "
        "Answer with a single word: YES or NO.\n\n```verilog\n"+src+"\n```\nAnswer:")
    text=zt.apply_chat_template([{"role":"user","content":pr}], tokenize=False, add_generation_prompt=True)
    enc=zt(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad(): out=zl.generate(**enc, max_new_tokens=3, do_sample=False)
    return 1 if zt.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper().startswith("Y") else 0
tp=fp=fn=tn=0
print(f"zero-shot over {len(test)} modules...")
for r in test:
    yt=int(r["label"]); yp=zs_pred(r)
    tp+=yt&yp; fp+=(yp and not yt); fn+=(yt and not yp); tn+=(not yt and not yp)
prec=tp/(tp+fp) if tp+fp else 0; rec=tp/(tp+fn) if tp+fn else 0
m_zs=dict(precision=round(prec,3), recall=round(rec,3), f1=round(2*prec*rec/(prec+rec),3) if prec+rec else 0.0, tp=tp,fp=fp,fn=fn,tn=tn)
print("ZERO-SHOT:", m_zs)

In [ ]:
# Cell 12 — summary
def row(n,m): return f"{n:<14}| F1 {m['f1']:.3f} | rec {m['recall']:.3f} | prec {m['precision']:.3f}"
print("="*60); print("Held-out test:", sorted(test_vars)); print("="*60)
print(row("Zero-shot", m_zs))
print(row("Frozen probe", m_probe), f"| PLC {m_probe['PLC']:.2f} IoU {m_probe['IoU']:.2f} VR {m_probe['VR']:.2f}")
print(row("Fine-tuned", m_ft), f"| PLC {m_ft['PLC']:.2f} IoU {m_ft['IoU']:.2f} VR {m_ft['VR']:.2f}")
print("="*60)
print(f"Faithfulness (fine-tuned): Delta_c {m_ft['delta_c']}, Delta_s {m_ft['delta_s']}, AG {m_ft['AG']}, VR {m_ft['VR']}")